# Multi-Modal & Text Retrieval System

Notebook này xây dựng **hai hệ thống truy xuất thông tin (retrieval systems)**:

- **Part A — Text-to-Image Retrieval**: tìm ảnh Flickr8k phù hợp nhất với một câu mô tả, dùng **OpenCLIP + FAISS**.
- **Part B — Text-to-Text Retrieval**: tìm tài liệu khoa học SciFact liên quan nhất tới một câu hỏi, dùng **Dense Retrieval (SentenceTransformer) + Cross-Encoder reranking + FAISS**.

Cấu hình mô hình/index bên dưới là cấu hình **tốt nhất đã được đo đạc và lựa chọn** trong quá trình thử nghiệm gốc (so sánh Flat/IVF/HNSW/PQ cho ảnh, và BM25/Dense/Dense+CrossEncoder/Hybrid-RRF cho văn bản). Notebook này chỉ giữ lại pipeline cuối cùng để dễ đọc, dễ chạy lại từ đầu đến cuối.

**Kết quả đã đo trong lần thử nghiệm gốc** (tham khảo, không hard-code lại — mọi số liệu trong notebook này đều được tính lại từ lần chạy hiện tại):

| Text-to-Image (FAISS index) | Recall@1 | Recall@5 | MRR |
|---|---|---|---|
| **Flat (đã chọn)** | ~0.3270 | ~0.5756 | ~0.4328 |
| HNSW | thấp hơn, nhanh hơn | | |
| PQ | recall giảm mạnh, RAM thấp | | |

| Text-to-Text system | Recall@10 | MRR |
|---|---|---|
| BM25 | ~70.67% | ~0.5292 |
| Dense MiniLM | ~80.00% | ~0.6068 |
| **Dense + Cross-Encoder (đã chọn)** | ~83.00% | ~0.6615 |
| Hybrid RRF | ~80.67% | ~0.6163 |

Dựa trên các số liệu này, pipeline cuối cùng dùng **FAISS `IndexFlatIP`** cho ảnh và **Dense retrieval + Cross-Encoder reranking** cho văn bản.

## 1. Setup

In [ ]:
%pip install -q open_clip_torch datasets pandas numpy pillow torch torchvision \
    pyarrow matplotlib faiss-cpu sentence-transformers


In [ ]:
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from datasets import load_dataset
import open_clip
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder


## 2. Configuration and Device

In [ ]:
# Tự động chọn thiết bị tính toán: ưu tiên CUDA, sau đó Apple Silicon (MPS), cuối cùng CPU
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

# --- Text-to-Image config ---
NUM_SAMPLES = 5000        # số ảnh Flickr8k dùng để xây index
IMAGE_BATCH_SIZE = 64     # batch size khi encode ảnh / text truy vấn ảnh
CLIP_MODEL_NAME = "RN50"  # cấu hình OpenCLIP tốt nhất đã chọn (xem bảng so sánh ở trên)
CLIP_PRETRAINED = "openai"

# --- Text-to-Text config ---
DENSE_MODEL_ID = "all-MiniLM-L6-v2"
RERANKER_MODEL_ID = "cross-encoder/ms-marco-MiniLM-L-6-v2"


## 3. Shared Utilities

In [ ]:
def _validate_query(query_text):
    """Validate a single query string and return it stripped."""
    if not isinstance(query_text, str) or not query_text.strip():
        raise ValueError("query_text must be a non-empty string")
    return query_text.strip()


def _validate_positive_int(value, name):
    """Validate that `value` is a positive integer."""
    if not isinstance(value, int) or value < 1:
        raise ValueError(f"{name} must be a positive integer")


def compute_retrieval_metrics(retrieved_ids, relevant_ids, k):
    """Compute Recall@K, Hit@K and Reciprocal Rank for one query.

    Recall@K = (# relevant docs retrieved in top-K) / (# relevant docs total).
    Hit@K = 1.0 if at least one relevant doc appears in top-K, else 0.0.
    Reciprocal Rank = 1 / rank of the first relevant doc in top-K (0.0 if none).
    """
    relevant_ids = set(relevant_ids)
    retrieved_at_k = list(retrieved_ids[:k])

    recall = (
        len(set(retrieved_at_k) & relevant_ids) / len(relevant_ids)
        if relevant_ids else 0.0
    )
    hit = float(any(doc_id in relevant_ids for doc_id in retrieved_at_k))
    first_relevant_rank = next(
        (rank for rank, doc_id in enumerate(retrieved_at_k, start=1)
         if doc_id in relevant_ids),
        None,
    )
    reciprocal_rank = 1.0 / first_relevant_rank if first_relevant_rank is not None else 0.0

    return {"recall": recall, "hit": hit, "reciprocal_rank": reciprocal_rank}


---
# Part A — Text-to-Image Retrieval

Pipeline: `Flickr8k images -> OpenCLIP image encoder -> normalized embeddings -> FAISS IndexFlatIP -> text query -> OpenCLIP text encoder -> normalized query -> FAISS search -> top-K images`

## 4. Load Flickr8k

In [ ]:
flickr_dataset = load_dataset("jxie/flickr8k", split="train")
flickr_dataset = flickr_dataset.select(range(NUM_SAMPLES))
print(f"Loaded {len(flickr_dataset)} Flickr8k samples.")


## 5. Load OpenCLIP

In [ ]:
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    CLIP_MODEL_NAME, pretrained=CLIP_PRETRAINED, device=device
)
clip_tokenizer = open_clip.get_tokenizer(CLIP_MODEL_NAME)
clip_model.eval()
print(f"OpenCLIP ready: {CLIP_MODEL_NAME} ({CLIP_PRETRAINED})")


In [ ]:
def encode_clip_texts(texts, batch_size=64):
    """Encode texts with OpenCLIP and return normalized float32 embeddings.

    Processes texts in batches so evaluation over thousands of captions does
    not require a single giant forward pass.
    """
    if not texts:
        raise ValueError("texts must not be empty")

    all_embeddings = []
    with torch.inference_mode():
        for start in range(0, len(texts), batch_size):
            batch = list(texts[start : start + batch_size])
            tokens = clip_tokenizer(batch).to(device)
            features = clip_model.encode_text(tokens)
            features = F.normalize(features, dim=-1)
            all_embeddings.append(features.cpu().numpy())

    return np.ascontiguousarray(np.vstack(all_embeddings), dtype=np.float32)


## 6. Encode Images

In [ ]:
# Encode toàn bộ ảnh theo batch:
# Dataset -> OpenCLIP image encoder -> Embedding -> L2 Normalize (bắt buộc để dùng IndexFlatIP như cosine similarity)
image_embeddings_list = []
image_metadata = []

with torch.inference_mode():
    for start in tqdm(range(0, len(flickr_dataset), IMAGE_BATCH_SIZE), desc="Encoding images"):
        batch = flickr_dataset[start : start + IMAGE_BATCH_SIZE]

        images_tensor = torch.stack(
            [clip_preprocess(img.convert("RGB")) for img in batch["image"]]
        ).to(device)

        features = clip_model.encode_image(images_tensor)
        features = F.normalize(features, dim=-1)  # L2 normalize -> inner product = cosine similarity
        image_embeddings_list.append(features.cpu().numpy())

        for offset, caption in enumerate(batch["caption_0"]):
            global_idx = start + offset
            image_metadata.append({
                "index": global_idx,
                "image_id": f"flickr8k_frame_{global_idx:04d}",
                "caption": caption,
            })

image_embeddings = np.ascontiguousarray(np.vstack(image_embeddings_list), dtype=np.float32)
image_metadata_df = pd.DataFrame(image_metadata)
print(f"Image embeddings shape: {image_embeddings.shape}")


## 7. Build FAISS Image Index

In [ ]:
# Trong thử nghiệm gốc, IndexFlatIP (exact search) cho Recall@1/@5/MRR cao nhất
# so với IVF, HNSW, PQ, nên được chọn làm index cuối cùng cho hệ thống.
embedding_dim = image_embeddings.shape[1]

image_index = faiss.IndexFlatIP(embedding_dim)
image_index.add(image_embeddings)
print(f"FAISS image index ready: {image_index.ntotal} vectors, dim={embedding_dim}")


## 8. Image Search

In [ ]:
def _plot_image_results(query_text, results):
    """Display a grid of retrieved images with their score and caption."""
    columns = min(len(results), 5)
    rows = int(np.ceil(len(results) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(4 * columns, 4 * rows))
    axes = np.atleast_1d(axes).ravel()

    for axis, result in zip(axes, results):
        item = flickr_dataset[result["index"]]
        axis.imshow(item["image"].convert("RGB"))
        caption = str(result["caption"])[:65]
        axis.set_title(f"score={result['score']:.3f}\n{caption}", fontsize=9)
        axis.axis("off")
    for axis in axes[len(results):]:
        axis.axis("off")

    fig.suptitle(f'Top {len(results)}: "{query_text}"', fontsize=13)
    plt.tight_layout()
    plt.show()


def search_images(query_text, k=5, show=True):
    """Retrieve the top-k Flickr8k images matching an English text query.

    Query -> OpenCLIP text encoder -> normalize -> FAISS search -> top-K images
    """
    query_text = _validate_query(query_text)
    _validate_positive_int(k, "k")

    result_count = min(k, image_index.ntotal)
    query_vector = encode_clip_texts([query_text], batch_size=1)

    scores, indices = image_index.search(query_vector, result_count)
    scores, indices = scores[0], indices[0]

    results = [
        {
            "index": int(idx),
            "score": float(score),
            "caption": image_metadata_df.loc[int(idx), "caption"],
        }
        for score, idx in zip(scores, indices)
    ]

    if show:
        _plot_image_results(query_text, results)

    return results


## 9. Image Retrieval Evaluation

In [ ]:
def evaluate_image_retrieval(index, k_values=(1, 5, 10), text_batch_size=IMAGE_BATCH_SIZE):
    """Evaluate a FAISS image index using each image's own caption as its query.

    Every query has exactly one correct image (its source), so this reports
    Recall@K and MRR over `len(image_metadata_df)` self-retrieval queries.
    """
    if index.ntotal == 0:
        raise ValueError("index must not be empty")
    if not k_values:
        raise ValueError("k_values must not be empty")
    for k in k_values:
        _validate_positive_int(k, "each value in k_values")

    # Giữ nguyên template truy vấn "a photo of {caption}" để kết quả có thể
    # so sánh trực tiếp với các phiên bản notebook trước.
    test_queries = [f"a photo of {caption}" for caption in image_metadata_df["caption"]]
    ground_truth = image_metadata_df["index"].tolist()

    text_features = encode_clip_texts(test_queries, batch_size=text_batch_size)

    max_k = min(max(k_values), index.ntotal)
    start = time.perf_counter()
    _, retrieved = index.search(text_features, max_k)
    search_total_ms = (time.perf_counter() - start) * 1000

    num_queries = len(test_queries)
    recalls = {k: 0 for k in k_values}
    mrr_sum = 0.0
    for q_idx in range(num_queries):
        target = ground_truth[q_idx]
        ranked = retrieved[q_idx]
        hit = np.where(ranked == target)[0]
        if len(hit):
            rank = hit[0] + 1
            mrr_sum += 1.0 / rank
            for k in k_values:
                if rank <= k:
                    recalls[k] += 1

    metrics = {f"Recall@{k}": recalls[k] / num_queries for k in k_values}
    metrics["MRR"] = mrr_sum / num_queries
    metrics["FAISS Search Total (ms)"] = search_total_ms
    metrics["FAISS Search Avg (ms/query)"] = search_total_ms / num_queries
    return metrics


**Lưu ý về latency:** phần đánh giá ảnh bên dưới chỉ đo thời gian gọi `index.search(...)` (FAISS search thuần), không tính thời gian encode text. Phần đánh giá văn bản ở Part B đo **toàn bộ pipeline** (encode query + FAISS search + rerank nếu có). Hai con số latency này **không tương đương** và không nên so sánh trực tiếp với nhau.

In [ ]:
image_metrics = evaluate_image_retrieval(image_index)
pd.DataFrame([image_metrics]).style.format({
    "Recall@1": "{:.4f}", "Recall@5": "{:.4f}", "Recall@10": "{:.4f}",
    "MRR": "{:.4f}",
    "FAISS Search Total (ms)": "{:.2f}",
    "FAISS Search Avg (ms/query)": "{:.4f}",
})


---
# Part B — Text-to-Text Retrieval

Pipeline: `SciFact corpus -> SentenceTransformer -> normalized embeddings -> FAISS IndexFlatIP -> query -> dense top-30 candidates -> Cross-Encoder rerank -> top-10 documents`

Trong thử nghiệm gốc, bốn hệ thống đã được so sánh: **BM25**, **Dense (MiniLM)**, **Dense + Cross-Encoder**, và **Hybrid RRF**. Dense + Cross-Encoder cho Recall@10 và MRR cao nhất (~83.00%, MRR ~0.6615), nên được chọn làm pipeline chính thức. BM25 và Hybrid RRF không được cài lại ở đây vì hiệu năng thấp hơn.

## 10. Load SciFact

In [ ]:
# SciFact: corpus (tài liệu), queries (câu hỏi), qrels (ground truth)
corpus_dataset = load_dataset("BeIR/scifact", "corpus", split="corpus")
queries_dataset = load_dataset("BeIR/scifact", "queries", split="queries")
qrels_dataset = load_dataset("BeIR/scifact-qrels", split="test")

corpus_dict = {str(item["_id"]): item["title"] + ". " + item["text"] for item in corpus_dataset}
corpus_ids = list(corpus_dict.keys())
corpus_texts = list(corpus_dict.values())

query_dict = {str(item["_id"]): item["text"] for item in queries_dataset}

# Một query có thể có nhiều tài liệu liên quan (relevant docs) trong ground truth
ground_truth = {}
for row in qrels_dataset:
    q_id, c_id = str(row["query-id"]), str(row["corpus-id"])
    if q_id in query_dict and c_id in corpus_dict:
        ground_truth.setdefault(q_id, []).append(c_id)

valid_q_ids = list(ground_truth.keys())  # chỉ giữ query có ground truth để đánh giá

print(f"Corpus: {len(corpus_dict)} documents")
print(f"Queries: {len(query_dict)} total, {len(valid_q_ids)} with ground truth")


## 11. Load Dense Model and CrossEncoder

In [ ]:
dense_model = SentenceTransformer(DENSE_MODEL_ID, device=device)
reranker = CrossEncoder(RERANKER_MODEL_ID, device=device)
print(f"Dense model: {DENSE_MODEL_ID}")
print(f"Reranker: {RERANKER_MODEL_ID}")


In [ ]:
def encode_dense_texts(texts, batch_size=64):
    """Encode texts with SentenceTransformer and return normalized float32 embeddings.

    Works for both corpus documents and query text.
    """
    if not texts:
        raise ValueError("texts must not be empty")

    embeddings = dense_model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=False,
        convert_to_numpy=True,
    )
    embeddings = np.ascontiguousarray(embeddings, dtype=np.float32)
    faiss.normalize_L2(embeddings)
    return embeddings


## 12. Encode Corpus and Build FAISS Index

In [ ]:
# Document -> SentenceTransformer -> Embedding -> L2 Normalize -> FAISS IndexFlatIP
corpus_embeddings = encode_dense_texts(corpus_texts, batch_size=64)

text_index = faiss.IndexFlatIP(corpus_embeddings.shape[1])
text_index.add(corpus_embeddings)
print(f"FAISS text index ready: {text_index.ntotal} documents, dim={corpus_embeddings.shape[1]}")


## 13. Dense Retrieval

In [ ]:
def dense_search(query_text, top_k=30):
    """Retrieve first-stage dense candidates from the SciFact FAISS index."""
    query_text = _validate_query(query_text)
    _validate_positive_int(top_k, "top_k")
    top_k = min(top_k, len(corpus_ids))

    query_vector = encode_dense_texts([query_text], batch_size=1)
    scores, indices = text_index.search(query_vector, top_k)

    return [
        (corpus_ids[int(idx)], float(score))
        for score, idx in zip(scores[0], indices[0])
        if idx >= 0
    ]


## 14. CrossEncoder Reranking

In [ ]:
def search_documents(query_text, top_k=10, candidate_k=30):
    """Retrieve dense candidates and rerank them with the CrossEncoder.

    Pipeline: dense top-`candidate_k` candidates -> CrossEncoder rerank -> top-`top_k`.
    """
    query_text = _validate_query(query_text)
    _validate_positive_int(top_k, "top_k")
    _validate_positive_int(candidate_k, "candidate_k")

    candidate_k = max(candidate_k, top_k)
    candidate_k = min(candidate_k, len(corpus_ids))
    top_k = min(top_k, candidate_k)

    candidates = dense_search(query_text, top_k=candidate_k)

    # CrossEncoder chấm điểm trực tiếp trên cặp (query, document text gốc)
    pairs = [(query_text, corpus_dict[doc_id]) for doc_id, _ in candidates]
    cross_scores = reranker.predict(pairs, batch_size=32, show_progress_bar=False)

    reranked = sorted(
        zip((doc_id for doc_id, _ in candidates), cross_scores),
        key=lambda item: -float(item[1]),
    )[:top_k]

    return [
        {"doc_id": doc_id, "score": float(score), "text": corpus_dict[doc_id]}
        for doc_id, score in reranked
    ]


## 15. Retrieval Metrics

In [ ]:
def evaluate_text_retrieval(search_fn, query_ids, top_k=10):
    """Evaluate a search callable with Recall@K, Hit@K, MRR and end-to-end latency.

    Latency here covers the full pipeline invoked by `search_fn` (query encoding
    + FAISS search, and CrossEncoder reranking when `search_fn` performs it).
    """
    if not query_ids:
        raise ValueError("query_ids must not be empty")

    recall_sum, hit_sum, rr_sum, elapsed_sum = 0.0, 0.0, 0.0, 0.0

    for query_id in query_ids:
        started = time.perf_counter()
        results = search_fn(query_dict[query_id], top_k=top_k)
        elapsed_sum += time.perf_counter() - started

        retrieved_ids = [r["doc_id"] if isinstance(r, dict) else r[0] for r in results]
        query_metrics = compute_retrieval_metrics(
            retrieved_ids, ground_truth[query_id], k=top_k
        )
        recall_sum += query_metrics["recall"]
        hit_sum += query_metrics["hit"]
        rr_sum += query_metrics["reciprocal_rank"]

    count = len(query_ids)
    return {
        f"Recall@{top_k}": 100.0 * recall_sum / count,
        f"Hit@{top_k}": 100.0 * hit_sum / count,
        "MRR": rr_sum / count,
        "Avg End-to-End Latency (ms/query)": 1000.0 * elapsed_sum / count,
    }


## 16. Text Retrieval Evaluation

In [ ]:
# So sánh Dense retrieval (chưa rerank) với Dense + Cross-Encoder (pipeline cuối cùng)
dense_only_metrics = evaluate_text_retrieval(
    lambda q, top_k: [{"doc_id": doc_id} for doc_id, _ in dense_search(q, top_k=top_k)],
    valid_q_ids, top_k=10,
)
dense_rerank_metrics = evaluate_text_retrieval(search_documents, valid_q_ids, top_k=10)

text_results_df = pd.DataFrame([
    {"System": "Dense (MiniLM only)", **dense_only_metrics},
    {"System": "Dense + Cross-Encoder", **dense_rerank_metrics},
])
text_results_df.style.format({
    "Recall@10": "{:.2f}%", "Hit@10": "{:.2f}%", "MRR": "{:.4f}",
    "Avg End-to-End Latency (ms/query)": "{:.2f}",
})


---
# Final Demo

In [ ]:
# Demo Text-to-Image
_ = search_images("a dog running through green grass", k=5)


In [ ]:
# Demo Text-to-Text: chọn một câu hỏi có ground truth từ SciFact để kết quả có ý nghĩa
demo_query_id = valid_q_ids[0]
demo_query_text = query_dict[demo_query_id]
print(f"Query: {demo_query_text}\n")

for result in search_documents(demo_query_text, top_k=10):
    print(f"[{result['score']:.3f}] {result['doc_id']}: {result['text'][:150]}...")
